# 07 · Static and adaptive V-optimal MIRT

Use a frozen teammate model, a supplied validation-chosen D, or the choice from 06.
Notebook 06 is optional when a model is supplied. Compare **random, MIRT anchors, static V-optimal and
adaptive V-optimal** with the same estimator. All accuracy / p-IRT / gp-IRT combinations are retained.

For our item loading vector $\alpha_i$, information is
$p_i(1-p_i)\alpha_i\alpha_i^\top$. It is a matrix.
We target mean **logit prediction variance across the full MMLU item population**, using
$M=Q^{-1}\sum_i\alpha_i\alpha_i^\top$.
For posterior covariance $C$, candidate $i$ reduces $\mathrm{tr}(MC)$ by

$$g_i=\frac{w_i\, (C\alpha_i)^\top M(C\alpha_i)}
{1+w_i\alpha_i^\top C\alpha_i},\qquad w_i=p_i(1-p_i).$$

This is [Krumdick et al., Eq. 6](https://arxiv.org/html/2604.01418v1#S3.SS2).
It is a local covariance approximation with item parameters treated as fixed, not a calibrated
score interval or an exact variance of the mean score.

Static: average information over training abilities, then greedily update covariance after
adding each item. Adaptive: start at the training mean, observe one answer, refit ability,
and recompute accumulated information at the current ability before choosing the next item.
Both include the training Gaussian prior precision and forbid repeat items.

In [ ]:
import sys

sys.path.insert(0, "../scripts/irt")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from efficbench import ROOT, load
from protocol import fit_context, make_splits
from run_progression import VOPTIMAL_SELECTORS, run_selection

load_data = load  # Swap the loader while keeping response / model / item column order aligned.
ks = [10, 20, 50, 100, 200, 500]
seeds = range(20)

Y, models, items = load_data()
Y = Y.astype(np.float32)
splits = make_splits(Y)
subjects, model_ids = items.subject.values, models.model_id.values
item_ids = items.item_id.values


# Supply one frozen adapter per split to use a teammate's fitted MIRT directly.
contexts = None
build_context = fit_context
n_dims = None  # For standalone fitting, set D already chosen using train/validation.
out = ROOT / "results/irt/progression/voptimal"

## Model plug-in point

These stages do not require our fitter. Set `contexts` (or `contexts_by_dim` in 06) to
**one training-only frozen model per family split**, in that split order. Or replace
`build_context(Y_train, subjects, n_dims=...)` with your teammate's fitter.

For fitted 1PL/2PL/MIRT parameters:

```python
from cabench.design import parameter_context
ctx = parameter_context(train_theta, item_alpha, item_beta, item_ids=item_ids)
# Optional: retain your teammate's ability estimator (responses contains ONLY selected answers).
ctx["infer"] = teammate_infer  # (responses, items, initial=None) -> (models, D)
```

`item_beta` is the intercept in `sigmoid(theta @ alpha.T - beta)`; if the fit uses
`sigmoid(a * (theta-b))`, pass `beta=a*b`. Use ones for Rasch discrimination.
For a different backend, use `model_context` with prediction / inference callbacks and fixed
loadings; non-logistic links also need their correct `information_weights` callback.
See `scripts/irt/README.md` for shapes and examples. Keep item IDs, latent coordinates,
training prior and response columns aligned. Use a distinct `out` directory per fitted model.
The experiment helpers perform no refit when supplied with contexts.

## Fit train, calibrate on validation, evaluate test

Same splits, 20 seeds and k = 10…500. The deterministic static/adaptive designs share their
trajectory across seeds; repeats keep pairing with the random/anchor runs, not extra evidence.

In [ ]:
if contexts is None:
    if n_dims is None:
        choice = ROOT / "results/irt/progression/mirt/chosen_dimension.csv"
        if not choice.exists():
            raise ValueError("Supply contexts or a validation-chosen n_dims; 06 is optional")
        n_dims = int(pd.read_csv(choice).n_dims.iloc[0])
    contexts = [build_context(Y[train], subjects, n_dims=n_dims) for train, _, _ in splits]
chosen_dims = contexts[0]["theta"].shape[1]
metrics, records = run_selection(
    Y, splits, contexts, VOPTIMAL_SELECTORS, out, model_ids, ks=ks, seeds=seeds, item_ids=item_ids
)

## Hold gp-IRT fixed across selectors

Mean ± one standard deviation across family splits, after averaging seeds within each split.

In [ ]:
gp = metrics[metrics.estimator == "gp-IRT"]
per_split = gp.groupby(["selector", "k", "split"]).mae.mean()
summary = per_split.groupby(["selector", "k"]).agg(["mean", "std"])
fig, ax = plt.subplots(figsize=(8, 4.5))
for selector in VOPTIMAL_SELECTORS:
    curve = summary.loc[selector]
    ax.errorbar(
        curve.index,
        100 * curve["mean"],
        yerr=100 * curve["std"],
        marker="o",
        capsize=3,
        label=selector,
    )
ax.set(
    xscale="log",
    xlabel="k (answers observed)",
    ylabel="test MAE (pp)",
    title=f"MIRT D={chosen_dims} + gp-IRT (±1 SD across splits)",
)
ax.legend()
plt.tight_layout()

In [ ]:
100 * metrics.groupby(["k", "selector", "estimator"]).mae.mean().unstack("estimator")

## Paired changes

Against random + gp-IRT, then adaptive against static V-optimal with gp-IRT.
Negative means lower MAE. Bootstrap split means after averaging seeds within models;
the five family splits overlap, so interpret the CI as split variability.

In [ ]:
from diagnostics import paired_comparison

against_random = paired_comparison(records)
against_static = paired_comparison(records, baseline="voptimal_static + gp-IRT")
headline = pd.concat(
    [
        against_random[against_random.method.str.endswith("+ gp-IRT")],
        against_static[against_static.method == "voptimal_adaptive + gp-IRT"],
    ],
    ignore_index=True,
)
headline[["delta_mae", "ci_low", "ci_high"]] *= 100
headline[["baseline", "method", "k", "delta_mae", "ci_low", "ci_high", "splits_won", "n_splits"]]

This is the endpoint of this implementation: single-benchmark scoring through V-optimality.
Cost-aware selection, cross-benchmark prediction and base→fine-tuned change analysis are deferred.